In [1]:
import pandas as pd
import numpy as np
import pickle, os, warnings
from datetime import timedelta
warnings.filterwarnings('ignore')

# ── UPDATE THESE PATHS ────────────────────────────────────────────────────────
RAW_DATA_PATH  = r"C:\Users\Rishit\Desktop\O2R-Order-prediction\data\Jan - May '26 Data.csv"
MODEL_PATH     = r"C:\Users\Rishit\Desktop\O2R-Order-prediction\models\xgboost_order_model.pkl"
ENCODER_PATH   = r"C:\Users\Rishit\Desktop\O2R-Order-prediction\processed\label_encoders.pkl"
PROFILE_PATH   = r"C:\Users\Rishit\Desktop\O2R-Order-prediction\processed\retailer_profiles.parquet"
OUTPUTS_DIR    = r"C:\Users\Rishit\Desktop\O2R-Order-prediction\outputs"
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUTS_DIR, exist_ok=True)
print('Setup complete.')

Setup complete.


In [2]:
with open(MODEL_PATH, 'rb') as f:
    saved = pickle.load(f)
model        = saved['model']
FEATURE_COLS = saved['feature_cols']
THRESHOLD    = saved['threshold']

with open(ENCODER_PATH, 'rb') as f:
    label_encoders = pickle.load(f)

retailer_profile = pd.read_parquet(PROFILE_PATH)

print(f'Model loaded. Trained on: {saved["trained_on"]} | Predicts for: {saved["predicts_for"]}')
print(f'Default threshold: {THRESHOLD}')
print(f'Features: {len(FEATURE_COLS)}')
print(f'Retailer profiles: {len(retailer_profile):,}')

Model loaded. Trained on: Jan-May 2026 | Predicts for: June 2026
Default threshold: 0.4
Features: 25
Retailer profiles: 8,640


In [3]:
print('Loading historical data...')
df = pd.read_csv(RAW_DATA_PATH)
df['createdAt'] = pd.to_datetime(df['createdAt'], dayfirst=True)

confirmed = df[df['orderStatus'].isin(['Delivered','PartiallyDelivered'])]
orders = confirmed.drop_duplicates(subset='orderNumber')[[
    'orderNumber','customerId','createdAt','hubName','shopType','retailerType','orderSource'
]].copy()

orders = orders.sort_values(['customerId','createdAt'])

HISTORY_END = orders['createdAt'].max()  # May 31 2026
print(f'History end date: {HISTORY_END.date()}')
print(f'Total historical orders: {len(orders):,}')

Loading historical data...
History end date: 2026-05-31
Total historical orders: 177,340


In [4]:
def build_june_features(orders, retailer_profile, label_encoders, target_date):
    """
    Build feature row for every retailer for a given June date.
    Uses only Jan-May history — no leakage.
    """
    target_date = pd.Timestamp(target_date)

    # Only use orders strictly before target date
    hist = orders[orders['createdAt'] < target_date].copy()

    # Last order date per retailer
    last_order = hist.groupby('customerId')['createdAt'].max().reset_index()
    last_order.columns = ['customerId', 'last_order_date']
    last_order['days_since_last_order'] = (
        target_date - last_order['last_order_date']
    ).dt.days

    # Rolling order counts (look back from target_date)
    def orders_in_last_n_days(n):
        cutoff = target_date - timedelta(days=n)
        return (
            hist[hist['createdAt'] >= cutoff]
            .groupby('customerId')['orderNumber'].count()
            .reset_index()
            .rename(columns={'orderNumber': f'orders_last_{n}_days'})
        )

    r3  = orders_in_last_n_days(3)
    r7  = orders_in_last_n_days(7)
    r14 = orders_in_last_n_days(14)
    r30 = orders_in_last_n_days(30)

    total_so_far = (
        hist.groupby('customerId')['orderNumber'].count()
        .reset_index()
        .rename(columns={'orderNumber': 'total_orders_so_far'})
    )

    # Gap stats per retailer
    hist_s = hist.sort_values(['customerId','createdAt'])
    hist_s['gap'] = hist_s.groupby('customerId')['createdAt'].diff().dt.days
    gap_stats = hist_s.groupby('customerId')['gap'].agg(
        avg_gap_between_orders = 'mean',
        std_gap_between_orders = 'std',
        median_gap             = 'median'
    ).reset_index()
    gap_stats = gap_stats.fillna({'avg_gap_between_orders': 30,
                                   'std_gap_between_orders': 0,
                                   'median_gap': 30})

    # App order ratio
    app_ratio = (
        hist.groupby('customerId')
        .apply(lambda x: (x['orderSource'] == 'App').mean())
        .reset_index()
    )
    app_ratio.columns = ['customerId', 'app_order_ratio']

    # Assemble base dataframe with all retailers
    all_retailers = retailer_profile[['customerId','hubName','shopType',
                                       'retailerType','tenure_days']].copy()

    features = all_retailers\
        .merge(last_order[['customerId','last_order_date','days_since_last_order']],
               on='customerId', how='left')\
        .merge(r3,           on='customerId', how='left')\
        .merge(r7,           on='customerId', how='left')\
        .merge(r14,          on='customerId', how='left')\
        .merge(r30,          on='customerId', how='left')\
        .merge(total_so_far, on='customerId', how='left')\
        .merge(gap_stats,    on='customerId', how='left')\
        .merge(app_ratio,    on='customerId', how='left')

    # Fill missing values (retailers with no recent history)
    features['days_since_last_order']  = features['days_since_last_order'].fillna(999)
    features['orders_last_3_days']     = features['orders_last_3_days'].fillna(0)
    features['orders_last_7_days']     = features['orders_last_7_days'].fillna(0)
    features['orders_last_14_days']    = features['orders_last_14_days'].fillna(0)
    features['orders_last_30_days']    = features['orders_last_30_days'].fillna(0)
    features['total_orders_so_far']    = features['total_orders_so_far'].fillna(0)
    features['avg_gap_between_orders'] = features['avg_gap_between_orders'].fillna(30)
    features['std_gap_between_orders'] = features['std_gap_between_orders'].fillna(0)
    features['median_gap']             = features['median_gap'].fillna(30)
    features['app_order_ratio']        = features['app_order_ratio'].fillna(0.5)
    features['tenure_days']            = features['tenure_days'].fillna(0)

    # Overdue features
    features['days_overdue'] = (
        features['days_since_last_order'] - features['avg_gap_between_orders']
    ).clip(lower=0)
    features['is_overdue']       = (features['days_overdue'] > 0).astype(int)
    features['order_regularity'] = 1 / (features['std_gap_between_orders'] + 1)
    features['overdue_ratio']    = (
        features['days_since_last_order'] / (features['avg_gap_between_orders'] + 1)
    ).clip(upper=10).round(3)

    # Temporal features
    features['date']           = target_date
    features['day_of_week']    = target_date.dayofweek
    features['day_of_month']   = target_date.day
    features['week_of_month']  = (target_date.day - 1) // 7 + 1
    features['month']          = target_date.month
    features['is_weekend']     = int(target_date.dayofweek >= 5)
    features['is_month_start'] = int(target_date.day <= 3)
    features['is_month_end']   = int(target_date.day >= 28)

    # Encode categoricals using saved encoders
    for col in ['hubName', 'shopType', 'retailerType']:
        le = label_encoders[col]
        known = set(le.classes_)
        features[col] = features[col].apply(
            lambda x: x if str(x) in known else le.classes_[0]
        )
        features[col + '_enc'] = le.transform(features[col].astype(str))

    return features

print('Feature builder function defined.')

Feature builder function defined.


In [5]:
# ── CHANGE THIS TO ANY JUNE DATE ──────────────────────────────────────────────
TARGET_DATE = '2026-06-01'
# ─────────────────────────────────────────────────────────────────────────────

print(f'Building features for: {TARGET_DATE}')
features = build_june_features(orders, retailer_profile, label_encoders, TARGET_DATE)
print(f'Feature rows built: {len(features):,}')

# Score
X_score = np.nan_to_num(features[FEATURE_COLS].values, nan=0.0)
features['order_probability'] = model.predict_proba(X_score)[:, 1]
features['will_order']        = (features['order_probability'] >= THRESHOLD).astype(int)

print(f'\nRetailers to CALL (prob ≥ {int(THRESHOLD*100)}%): {features["will_order"].sum():,}')
print(f'Retailers to SKIP                          : {(features["will_order"]==0).sum():,}')
print(f'Estimated call reduction                   : {(1 - features["will_order"].mean())*100:.1f}%')

Building features for: 2026-06-01
Feature rows built: 8,640

Retailers to CALL (prob ≥ 40%): 3,295
Retailers to SKIP                          : 5,345
Estimated call reduction                   : 61.9%


In [6]:
output = features[[
    'customerId','hubName','shopType','retailerType',
    'order_probability','will_order',
    'last_order_date','days_since_last_order',
    'avg_gap_between_orders','days_overdue','orders_last_7_days',
    'total_orders_so_far','app_order_ratio'
]].copy()

output['order_probability_pct'] = (output['order_probability'] * 100).round(1)
output['avg_gap_between_orders'] = output['avg_gap_between_orders'].round(1)
output['days_overdue'] = output['days_overdue'].round(1)

if 'last_order_date' in output.columns:
    output['last_order_date'] = pd.to_datetime(output['last_order_date']).dt.date

output = output.sort_values('order_probability', ascending=False).reset_index(drop=True)
output.index += 1
output.index.name = 'Rank'

print(f'=== TOP 20 RETAILERS TO CALL ON {TARGET_DATE} ===')
print(output[['customerId','hubName','shopType','order_probability_pct',
              'days_since_last_order','avg_gap_between_orders','days_overdue']].head(20).to_string())

=== TOP 20 RETAILERS TO CALL ON 2026-06-01 ===
      customerId                        hubName   shopType  order_probability_pct  days_since_last_order  avg_gap_between_orders  days_overdue
Rank                                                                                                                                          
1      USR-43064     Cross Line Events (EDelhi)  General A              97.800003                      2                     1.3           0.7
2     USR-151808      Crossline Events (Meerut)  General A              97.699997                      2                     1.3           0.7
3      USR-94731            Instant Foods (SED)  General A              97.699997                      2                     1.4           0.6
4      USR-18654           Instant Foods(Noida)     Paan B              97.599998                      2                     1.3           0.7
5      USR-96305     Cross Line Events (EDelhi)  General B              97.500000              

In [7]:
out_path = os.path.join(OUTPUTS_DIR, f'call_priority_{TARGET_DATE}.csv')
output.reset_index().to_csv(out_path, index=False)
print(f'Priority list saved → {out_path}')
print(f'Total retailers: {len(output):,}')
print(f'Call list size : {output["will_order"].sum():,}')

Priority list saved → C:\Users\Rishit\Desktop\O2R-Order-prediction\outputs\call_priority_2026-06-01.csv
Total retailers: 8,640
Call list size : 3,295


In [8]:
print('Generating predictions for all June 2026 dates...')
print('(This runs the model 30 times — may take 5-10 min)')

june_dates = pd.date_range('2026-06-01', '2026-06-30', freq='D')
june_summary = []

for date in june_dates:
    f = build_june_features(orders, retailer_profile, label_encoders, date)
    X = np.nan_to_num(f[FEATURE_COLS].values, nan=0.0)
    probs = model.predict_proba(X)[:, 1]
    calls = int((probs >= THRESHOLD).sum())
    june_summary.append({
        'date'         : date.date(),
        'day'          : date.strftime('%A'),
        'calls_needed' : calls,
        'skipped'      : len(f) - calls,
        'call_reduction': round((1 - calls/len(f))*100, 1)
    })
    print(f'  {date.date()} ({date.strftime("%a")}) → Call {calls:,} retailers')

june_df = pd.DataFrame(june_summary)
june_path = os.path.join(OUTPUTS_DIR, 'june_2026_call_schedule.csv')
june_df.to_csv(june_path, index=False)

print(f'\nFull June schedule saved → {june_path}')
print(f'\n=== JUNE 2026 SUMMARY ===')
print(f'Avg daily calls needed : {june_df["calls_needed"].mean():.0f}')
print(f'Avg call reduction     : {june_df["call_reduction"].mean():.1f}%')
print(june_df.to_string(index=False))

Generating predictions for all June 2026 dates...
(This runs the model 30 times — may take 5-10 min)
  2026-06-01 (Mon) → Call 3,295 retailers
  2026-06-02 (Tue) → Call 3,184 retailers
  2026-06-03 (Wed) → Call 2,976 retailers
  2026-06-04 (Thu) → Call 2,872 retailers
  2026-06-05 (Fri) → Call 2,953 retailers
  2026-06-06 (Sat) → Call 2,850 retailers
  2026-06-07 (Sun) → Call 77 retailers
  2026-06-08 (Mon) → Call 2,950 retailers
  2026-06-09 (Tue) → Call 2,805 retailers
  2026-06-10 (Wed) → Call 2,665 retailers
  2026-06-11 (Thu) → Call 2,512 retailers
  2026-06-12 (Fri) → Call 2,393 retailers
  2026-06-13 (Sat) → Call 2,298 retailers
  2026-06-14 (Sun) → Call 1 retailers
  2026-06-15 (Mon) → Call 2,256 retailers
  2026-06-16 (Tue) → Call 2,189 retailers
  2026-06-17 (Wed) → Call 2,154 retailers
  2026-06-18 (Thu) → Call 2,193 retailers
  2026-06-19 (Fri) → Call 2,257 retailers
  2026-06-20 (Sat) → Call 2,337 retailers
  2026-06-21 (Sun) → Call 5 retailers
  2026-06-22 (Mon) → Call 2,